---
title: Use Open Climate Service with the openEO Python client
short_title: openEO Python client
---

Open Climate Service implements the [openEO](https://openeo.org/) API — that is what the `open-climate-service` client uses under the hood. So as well as the `ClimateService` client shown in the other notebooks, you can point the **standard [openEO Python client](https://openeo.org/documentation/1.0/python/)** at an instance.

This is useful if you already know openEO, or want process graphs that are **portable** — the same code runs against any openEO backend. The client builds process graphs lazily and runs them on the server, so only the (small) result comes back.

Install the client with `pip install openeo`. You also need a running Open Climate Service instance (see the [section intro](intro.md)).

## 1) Connect

No authentication is required for a typical local or country deployment.

In [ ]:
import openeo

# Replace with the URL of your Open Climate Service instance
connection = openeo.connect("https://my-instance.example.org")
print("openEO API version:", connection.capabilities().api_version())

## 2) Browse collections

Published datasets are exposed as openEO **collections** (the same ones `ClimateService.datasets()` lists).

In [ ]:
for c in connection.list_collections():
    print(c["id"], "—", c.get("title", ""))

## 3) Load a collection as a data cube

`load_collection` describes the data you want — a collection, a bounding box, and a time range. Nothing runs yet; the cube is a lazy description of the computation.

In [ ]:
cube = connection.load_collection(
    "era5land_temperature_monthly",   # a published collection id from the list above
    spatial_extent={"west": 28.8, "south": -2.9, "east": 30.9, "north": -1.0},   # example: Rwanda
    temporal_extent=["2025-01-01", "2025-12-31"],
)

## 4) Build a process graph

Chain openEO processes to describe the result. Here we reduce the time dimension to a per-pixel mean over the period. The work happens server-side when we run it.

In [ ]:
# Mean over the time dimension ("t" in Open Climate Service cubes)
mean_temperature = cube.reduce_dimension(reducer="mean", dimension="t")

# Inspect the process graph that will be sent to the server
mean_temperature.flat_graph()

## 5) Run it

For small results and concrete export formats (GeoTIFF, NetCDF, PNG, CSV), run synchronously with `download()` — this returns the finished file. (Datacube/Zarr output is served via batch jobs instead; see below.)

In [ ]:
mean_temperature.download("mean-temperature-2025.tif", format="GTiff")

For long-running computations, submit a **batch job** instead, then poll and fetch the results — the production pattern:

In [ ]:
job = mean_temperature.create_job(title="mean-temperature-2025", format="GTiff")
job.start_and_wait()
results = job.get_results()
results.get_assets()

## Standard openEO — and beyond

Everything above is standard openEO, so the same process graph runs against any openEO backend.

Open Climate Service also ships **higher-level workflows** (stored process graphs) for common DHIS2 tasks — for example aggregating a dataset to organisation units and exporting a DHIS2 `dataValueSet` or a Chap CSV. Those are easiest to call with `ClimateService.execute()`:

- [Aggregate to organisation units](aggregate-to-org-units.ipynb)
- [Prepare data for Chap](prepare-data-for-chap.ipynb)

See the [openEO Python client documentation](https://openeo.org/documentation/1.0/python/) for the full API.